## Hyperparameter Tuning: Propagation Scope (σ)

The propagation scope parameter σ (`sigma_scope`) controls how far the Gaussian-kernel
influence of a matched question word propagates to neighbouring answer positions.

**From the paper (Section 4.2):**
> "We vary σ from 5 to 55 with a step of 10 and select the value that gives
> the best MAP on the development set."

We evaluate σ ∈ {5, 15, 25, 35, 45, 55}, train a fresh RNN-POA model for each
value on the training set, and evaluate on the dev set.  The best σ is chosen
based on the **highest dev MAP**.

In [ ]:
HIDDEN_DIM  = 50
SIGMA_SCOPE = 25
SIGMA_PRIME = 0.1
N_EPOCHS    = 30
PATIENCE    = 5
LR          = 1.0

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Hyperparameter Tuning: Propagation Scope (σ)
# ═══════════════════════════════════════════════════════════════

SIGMA_VALUES = [5, 15, 25, 35, 45, 55]

# We'll tune on WikiQA
sigma_results = []     # list of dicts: {sigma, map, mrr}

for sigma_val in SIGMA_VALUES:
    print(f'\n{"="*60}')
    print(f'  Training RNN-POA with σ = {sigma_val}')
    print(f'{"="*60}')

    # ── Reset seeds for fair comparison ──
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True

    # ── Create a fresh model with this sigma ──
    model_sigma = RNNPOA(
        embedding_matrix,
        hidden_dim=HIDDEN_DIM,
        sigma_scope=sigma_val,
        sigma_prime=SIGMA_PRIME,
    ).to(device)

    param_count = sum(p.numel() for p in model_sigma.parameters() if p.requires_grad)
    print(f'  Parameters: {param_count:,}')

    # ── Train ──
    history_sigma = train_model(
        model_sigma,
        wiki_train_loader,
        wiki_dev_loader,
        n_epochs=N_EPOCHS,
        patience=PATIENCE,
        lr=LR,
        save_name=f'best_wiki_poa_sigma{sigma_val}.pt',
    )

    # ── Evaluate on dev set ──
    dev_loss, dev_map, dev_mrr = evaluate(model_sigma, wiki_dev_loader)
    print(f'  σ={sigma_val}  =>  Dev MAP: {dev_map:.4f}  |  Dev MRR: {dev_mrr:.4f}')

    sigma_results.append({
        'sigma': sigma_val,
        'map': dev_map,
        'mrr': dev_mrr,
    })

    # Free GPU memory
    del model_sigma
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('  Sigma Tuning Complete')
print(f'{"="*60}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Sigma Tuning Results Table & Best Sigma Selection
# ═══════════════════════════════════════════════════════════════

print(f'\n{"="*60}')
print('  Sigma Tuning Results (WikiQA Dev Set)')
print(f'{"="*60}')
print(f'  {"σ (sigma)":>10}  {"MAP":>10}  {"MRR":>10}')
print(f'  {"-"*34}')

for entry in sigma_results:
    print(f'  {entry["sigma"]:>10}  {entry["map"]:>10.4f}  {entry["mrr"]:>10.4f}')

# ── Select the best sigma based on highest MAP ──
best_entry = max(sigma_results, key=lambda x: x['map'])
best_sigma = best_entry['sigma']
best_map   = best_entry['map']
best_mrr   = best_entry['mrr']

print(f'\n  ★ Best σ = {best_sigma}  (MAP = {best_map:.4f}, MRR = {best_mrr:.4f})')
print(f'{"="*60}')

# Update the global SIGMA_SCOPE so subsequent training uses the best value
SIGMA_SCOPE = best_sigma
print(f'\n  SIGMA_SCOPE updated to {SIGMA_SCOPE} for subsequent experiments.')

## 7. Training the Models

We train both the **RNN-POA** model and the **Attention-BLSTM baseline** for comparison.

In [ ]:
print('='*60)
print('Training RNN-POA (Positional Attention)')
print('='*60)
wiki_model_poa = RNNPOA(
    embedding_matrix, hidden_dim=HIDDEN_DIM,
    sigma_scope=SIGMA_SCOPE, sigma_prime=SIGMA_PRIME
).to(device)
print(f'Parameters: {sum(p.numel() for p in wiki_model_poa.parameters() if p.requires_grad):,}')

wiki_history_poa = train_model(wiki_model_poa, wiki_train_loader, wiki_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_wiki_poa.pt')

# Reset seeds
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('EXPERIMENT 2: TREC-QA (Clean)')

trec_model_poa = RNNPOA(embedding_matrix, hidden_dim=HIDDEN_DIM,
    sigma_scope=SIGMA_SCOPE, sigma_prime=SIGMA_PRIME).to(device)
trec_history_poa = train_model(trec_model_poa, trec_train_loader, trec_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_trec_poa.pt')

trec_model_base = AttentionBLSTM(embedding_matrix, hidden_dim=HIDDEN_DIM).to(device)
trec_history_base = train_model(trec_model_base, trec_train_loader, trec_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_trec_base.pt')

_, trec_test_map_poa, trec_test_mrr_poa = evaluate(trec_model_poa, trec_test_loader)
_, trec_test_map_base, trec_test_mrr_base = evaluate(trec_model_base, trec_test_loader)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Training RNN-ATT Baseline (Average Pooling, no attention)
# ═══════════════════════════════════════════════════════════════

# ── Reset seeds ──
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ── WikiQA ──
print('='*60)
print('Training RNN-ATT Baseline (WikiQA)')
print('='*60)

wiki_model_att = AttentionBLSTM(
    embedding_matrix, hidden_dim=HIDDEN_DIM
).to(device)
print(f'Parameters: {sum(p.numel() for p in wiki_model_att.parameters() if p.requires_grad):,}')

wiki_history_att = train_model(
    wiki_model_att, wiki_train_loader, wiki_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_wiki_att.pt'
)

# ── TREC-QA ──
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print('='*60)
print('Training RNN-ATT Baseline (TREC-QA)')
print('='*60)

trec_model_att = AttentionBLSTM(
    embedding_matrix, hidden_dim=HIDDEN_DIM
).to(device)

trec_history_att = train_model(
    trec_model_att, trec_train_loader, trec_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_trec_att.pt'
)

# ── Evaluate on test sets ──
_, wiki_test_map_att, wiki_test_mrr_att = evaluate(wiki_model_att, wiki_test_loader)
_, trec_test_map_att, trec_test_mrr_att = evaluate(trec_model_att, trec_test_loader)

print(f'\nRNN-ATT WikiQA Test  => MAP: {wiki_test_map_att:.4f}  MRR: {wiki_test_mrr_att:.4f}')
print(f'RNN-ATT TREC-QA Test => MAP: {trec_test_map_att:.4f}  MRR: {trec_test_mrr_att:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Training RNN-AVG Baseline (Average Pooling, no attention)
# ═══════════════════════════════════════════════════════════════

# ── Reset seeds ──
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ── WikiQA ──
print('='*60)
print('Training RNN-AVG Baseline (WikiQA)')
print('='*60)

wiki_model_avg = AvgPoolBLSTM(
    embedding_matrix, hidden_dim=HIDDEN_DIM
).to(device)
print(f'Parameters: {sum(p.numel() for p in wiki_model_avg.parameters() if p.requires_grad):,}')

wiki_history_avg = train_model(
    wiki_model_avg, wiki_train_loader, wiki_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_wiki_avg.pt'
)

# ── TREC-QA ──
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print('='*60)
print('Training RNN-AVG Baseline (TREC-QA)')
print('='*60)

trec_model_avg = AvgPoolBLSTM(
    embedding_matrix, hidden_dim=HIDDEN_DIM
).to(device)

trec_history_avg = train_model(
    trec_model_avg, trec_train_loader, trec_dev_loader,
    N_EPOCHS, PATIENCE, LR, save_name='best_trec_avg.pt'
)

# ── Evaluate on test sets ──
_, wiki_test_map_avg, wiki_test_mrr_avg = evaluate(wiki_model_avg, wiki_test_loader)
_, trec_test_map_avg, trec_test_mrr_avg = evaluate(trec_model_avg, trec_test_loader)

print(f'\nRNN-AVG WikiQA Test  => MAP: {wiki_test_map_avg:.4f}  MRR: {wiki_test_mrr_avg:.4f}')
print(f'RNN-AVG TREC-QA Test => MAP: {trec_test_map_avg:.4f}  MRR: {trec_test_mrr_avg:.4f}')

### Test Set Evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  3-Model Comparison Table: RNN-AVG vs RNN-ATT vs RNN-POA
# ═══════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('  FULL MODEL COMPARISON')
print('='*70)

# ── WikiQA ──
print('\n  WikiQA Test Results')
print('  ' + '-'*50)
print(f'  {"Model":<30} {"MAP":>8} {"MRR":>8}')
print('  ' + '-'*50)
print(f'  {"RNN-AVG (avg pool)":<30} {wiki_test_map_avg:>8.4f} {wiki_test_mrr_avg:>8.4f}')
print(f'  {"RNN-ATT (attention)":<30} {wiki_test_map_base:>8.4f} {wiki_test_mrr_base:>8.4f}')
print(f'  {"RNN-POA (positional attn)":<30} {wiki_test_map_poa:>8.4f} {wiki_test_mrr_poa:>8.4f}')

# Improvement analysis
if wiki_test_map_avg > 0:
    att_vs_avg_map = (wiki_test_map_base - wiki_test_map_avg) / wiki_test_map_avg * 100
    poa_vs_avg_map = (wiki_test_map_poa - wiki_test_map_avg) / wiki_test_map_avg * 100
    poa_vs_att_map = (wiki_test_map_poa - wiki_test_map_base) / wiki_test_map_base * 100
    print(f'\n  WikiQA MAP improvements:')
    print(f'    ATT vs AVG: {att_vs_avg_map:+.2f}%')
    print(f'    POA vs AVG: {poa_vs_avg_map:+.2f}%')
    print(f'    POA vs ATT: {poa_vs_att_map:+.2f}%')

# ── TREC-QA ──
print('\n  ' + '-'*50)
print('\n  TREC-QA Test Results')
print('  ' + '-'*50)
print(f'  {"Model":<30} {"MAP":>8} {"MRR":>8}')
print('  ' + '-'*50)
print(f'  {"RNN-AVG (avg pool)":<30} {trec_test_map_avg:>8.4f} {trec_test_mrr_avg:>8.4f}')
print(f'  {"RNN-ATT (attention)":<30} {trec_test_map_base:>8.4f} {trec_test_mrr_base:>8.4f}')
print(f'  {"RNN-POA (positional attn)":<30} {trec_test_map_poa:>8.4f} {trec_test_mrr_poa:>8.4f}')

if trec_test_map_avg > 0:
    att_vs_avg_map = (trec_test_map_base - trec_test_map_avg) / trec_test_map_avg * 100
    poa_vs_avg_map = (trec_test_map_poa - trec_test_map_avg) / trec_test_map_avg * 100
    poa_vs_att_map = (trec_test_map_poa - trec_test_map_base) / trec_test_map_base * 100
    print(f'\n  TREC-QA MAP improvements:')
    print(f'    ATT vs AVG: {att_vs_avg_map:+.2f}%')
    print(f'    POA vs AVG: {poa_vs_avg_map:+.2f}%')
    print(f'    POA vs ATT: {poa_vs_att_map:+.2f}%')

print('\n' + '='*70)
print('  Expected ranking: RNN-AVG < RNN-ATT < RNN-POA')
print('  This demonstrates the incremental benefit of attention,')
print('  and then positional attention, over simple average pooling.')
print('='*70)